In [99]:
import pandas as pd

products = pd.read_csv("../data/books.csv")
products.head()

,id,title,author,isbn,parent_genre,genre,sub_genre,price,pages,publisher,year,description
0,0,Where the Crawdads Sing,Delia Owens,9781043321818,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,3.56,422,Penguin Books,1954,Imprescindible para cualquier amante de la lec...
1,1,"Girl on the Train, The (RED)",Paula Hawkins,9780133890834,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,3.67,380,Random House,1967,"Narración brillante que combina suspense, emoc..."
2,2,"THE SILENT PATIENT [Paperback] Michaelides, Alex",Alex Michaelides,9787940265426,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,2.98,272,Macmillan Publishers,1968,"Escrita con una prosa elegante, esta obra es u..."
3,3,"The Silent Patient: The record-breaking, multi...",Alex Michaelides,9781615594079,Lifestyle & Leisure,"Arts, Film & Photography",Theory & Criticism,2.14,284,Macmillan Publishers,2016,Un viaje apasionante a través de ideas que tra...
4,4,"THE SILENT PATIENT [Paperback] Michaelides, Alex",Alex Michaelides,9781849593100,Fiction,Literature & Fiction,"Crime, Thriller & Mystery",2.98,703,Penguin Books,1986,Un viaje apasionante a través de ideas que tra...


In [100]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity


class ItemRecommendationEngine():

    def preprocess(
        self,
        df: pd.DataFrame,
    ) -> pd.DataFrame:

        pipeline = ColumnTransformer([
            (
                "cat",
                OneHotEncoder(
                    sparse_output=False,
                    handle_unknown="ignore",
                ),
                df.select_dtypes(
                    include=["str", "object", "category"],
                ).columns.tolist()
            ),
            (
                "num",
                MinMaxScaler(),
                df.select_dtypes(
                    include="number",
                ).columns.tolist(),
            ),
        ])

        return pd.DataFrame(
            pipeline.fit_transform(df),
            index=df.index,
        )

    def train(
        self,
        data: pd.DataFrame,
    ):
        features = self.preprocess(data.drop(columns="id"))

        sim = pd.DataFrame(
            cosine_similarity(features.values),
            index=data.set_index([i for i in data.columns]).index,
            columns=data["id"].values,
        )
        self.similarity_df = sim

        return self

    def predict(
        self,
        sample: pd.DataFrame,
        limit=6,
    ):
        if self.similarity_df is None:
            return None

        product_id = sample["id"].item()

        if product_id not in self.similarity_df.index:
            return None

        scores = (
            self.similarity_df[product_id]
            .drop(product_id)
            .nlargest(limit)
            .to_frame("similarity")
        )

        return scores


In [101]:
engine = ItemRecommendationEngine().train(products)

In [102]:
harry_potter = products.iloc[[107]]
harry_potter

,id,title,author,isbn,parent_genre,genre,sub_genre,price,pages,publisher,year,description
107,107,"Harry Potter and the Deathly Hallows, Book 7",J.K. Rowling,9785891783909,Children & Young Adult,Children's Books,Growing Up & Facts of Life,17.59,871,Scholastic,1981,Un viaje apasionante a través de ideas que tra...


In [104]:
data = engine.predict(harry_potter)
data

,,,,,,,,,,,,similarity
id,title,author,isbn,parent_genre,genre,sub_genre,price,pages,publisher,year,description,
74,"Harry Potter and the Philosopher's Stone, Book 1",J.K. Rowling,9786488771904,Children & Young Adult,Children's Books,Growing Up & Facts of Life,10.99,683,Oxford University Press,1992,Repleta de personajes complejos y giros inesperados que no te dejarán soltar el libro.,0.630819
341,Harry Potter Box Set: The Complete Collection (Children’s Paperback),J.K. Rowling,9781684602236,Children & Young Adult,Children's Books,Fantasy,30.64,915,Hachette Livre,1985,Un viaje apasionante a través de ideas que transforman la manera de ver el mundo.,0.629512
183,Harry Potter and the Half-Blood Prince,J.K. Rowling,9787858549733,Children & Young Adult,Children's Books,"Fantasy, Science Fiction & Horror",3.45,1094,Scholastic,1963,"Cautivadora desde el primer capítulo, con un desenlace que sorprende y conmueve.",0.622691
618,Matilda: Special Edition,Roald Dahl,9785196106465,Children & Young Adult,Children's Books,Growing Up & Facts of Life,2.85,981,Scholastic,1989,"Un clásico moderno que mezcla aventura, drama y crítica social de forma magistral.",0.622221
213,"Harry Potter and the Order of the Phoenix, Book 5",J.K. Rowling,9783936385465,Children & Young Adult,Children's Books,Growing Up & Facts of Life,17.59,561,Random House,2022,Un relato profundo y emotivo que explora la condición humana con maestría literaria.,0.621700
725,Diary of a Wimpy Kid: The Ugly Truth [Paperback] Jeff Kinney,Jeff Kinney,9784492683156,Children & Young Adult,Children's Books,"Fantasy, Science Fiction & Horror",2.61,1114,Scholastic,1978,Un viaje apasionante a través de ideas que transforman la manera de ver el mundo.,0.619168
